# 01 — Dos PDFs aos vetores que representam trechos

No notebook 00, uma pergunta chegou ao Qwen, mas sem documentos para consultar. Agora será criada a matéria-prima do RAG: texto extraído de PDFs, com fonte e página preservadas, dividido em trechos e transformado em embeddings. O resultado será salvo em disco para a etapa 02 construir o índice sem reextrair tudo.

```mermaid
flowchart LR
 A[PDFs + catálogo] --> B[texto por página]
 B --> C[chunks de até 96 tokens]
 C --> D[encoder multilíngue]
 D --> E[vetores normalizados + metadados]
 E --> F[dados_processados/]
```

In [1]:
# Objetivo: definir diretórios do corpus.
from pathlib import Path
import json
import hashlib

RANDOM_STATE = 42
DIRETORIO_PROJETO = Path.cwd()
if DIRETORIO_PROJETO.name != "Agente_IA_Transformers_Credito":
    DIRETORIO_PROJETO = DIRETORIO_PROJETO / "Agente_IA_Transformers_Credito"
if not DIRETORIO_PROJETO.is_dir():
    raise FileNotFoundError("Inicie o kernel na raiz do repositório ou na pasta do projeto.")

DIRETORIO_DOCUMENTOS = DIRETORIO_PROJETO / "documentos"
DIRETORIO_PROCESSADOS = DIRETORIO_PROJETO / "dados_processados"
DIRETORIO_RESULTADOS = DIRETORIO_PROJETO / "resultados"
DIRETORIO_PROCESSADOS.mkdir(exist_ok=True)
DIRETORIO_RESULTADOS.mkdir(exist_ok=True)
print("Projeto:", DIRETORIO_PROJETO)

Projeto: d:\Projetos_v2\Agente_IA_Transformers_Credito


## 1. Conhecer as fontes antes de indexá-las

Os arquivos em `documentos/` são o corpus documental que o assistente poderá consultar. O catálogo associa cada PDF à sua procedência e ao hash do arquivo, permitindo verificar qual versão foi usada. O conjunto inicial cobre fundamentos de atenção, inadimplência de consumidores e sequências de eventos bancários; cobertura sobre crédito brasileiro PF continua limitada.

A extração preserva o número da página do PDF para que cada trecho possa ser rastreado. Texto extraído não garante que fórmulas, tabelas ou colunas tenham mantido sua estrutura. Por isso, antes de tratar uma passagem como evidência, deve-se conferir o PDF original. Os documentos não são automaticamente convertidos em exemplos de treinamento.

---

### Extração de PDF: exemplo de risco

Em um PDF com duas colunas, a extração pode ler primeiro a linha inteira da esquerda e depois a da direita, ou alternar trechos entre colunas. Uma frase como “a janela de performance é de 12 meses” pode ficar separada de sua continuação. Preservar o número da página permite voltar ao original e corrigir ou descartar um trecho mal extraído.

In [2]:
# Objetivo: extrair texto dos PDFs com rastreabilidade.
from pypdf import PdfReader
import hashlib
catalogo = json.loads((DIRETORIO_PROJETO / "documentos/catalogo.json").read_text(encoding="utf-8"))
paginas = []
for fonte in catalogo:
    arquivo = DIRETORIO_PROJETO / "documentos" / fonte["file"]
    assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == fonte["sha256"]
    # PdfReader abre o PDF pelo caminho e disponibiliza suas páginas.
    leitor = PdfReader(arquivo)
    print(fonte["title"], "->", len(leitor.pages), "páginas")
    for numero, pagina in enumerate(leitor.pages, start=1):
        # extract_text tenta extrair texto da página; páginas sem texto viram string vazia.
        texto = pagina.extract_text() or ""
        if texto.strip():
            paginas.append({"texto": texto, "source_id": fonte["id"], "title": fonte["title"], "page": numero, "url": fonte["url"]})
print("Páginas com texto:", len(paginas))
print(paginas[0]["texto"][:1200])

Attention Is All You Need -> 15 páginas
Predicting Consumer Default: A Deep Learning Approach -> 68 páginas
PRAGMA: Revolut Foundation Model -> 20 páginas
Páginas com texto: 103
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an att

## 2. Dividir em chunks e transformá-los em embeddings

O encoder usado para busca tem um limite de contexto menor que um artigo inteiro. Antes de calcular vetores, cada página é dividida em janelas curtas, para que o texto enviado ao encoder não seja truncado silenciosamente. Neste experimento, cada janela tem até 96 tokens do tokenizer do encoder, e janelas vizinhas compartilham 16 tokens.

```text
Página longa → [tokens 1–96] → [tokens 81–176] → [tokens 161–256] → …
                         └── 16 tokens repetidos ──┘
```

A sobreposição ajuda a preservar uma ideia que cruza a fronteira entre janelas, mas aumenta a redundância. Ela será relevante ao estudar MMR na etapa 02. Como a divisão é por tokens, uma frase pode ser cortada; por isso, os chunks recuperados serão inspecionados.

Cada chunk passa então pelo encoder e vira um vetor numérico. Esse embedding representa o texto para a busca; ele não é o vetor de entrada do Qwen. Os vetores são normalizados para norma L2 igual a 1, o que permite usar produto escalar como cosseno na etapa seguinte. É essencial manter o alinhamento: o vetor de índice i deve continuar associado ao texto, à página e à fonte de índice i.

---

### Exemplo numérico de janelas

Com tamanho máximo de 96 tokens e sobreposição de 16, o avanço entre inícios é 96 − 16 = 80 tokens. Para uma página de 256 tokens, os inícios ficam em 0, 80, 160 e 240: formam-se janelas com 96, 96, 96 e 16 tokens. Os 16 tokens repetidos carregam contexto entre uma janela e a próxima; em contrapartida, geram redundância no índice.

### Embedding e normalização: exemplo pequeno

Um embedding representa um texto como um vetor numérico. No notebook, cada trecho de documento será convertido em um vetor, permitindo compará-lo com o vetor de uma pergunta.

Para compreender a normalização, considere o vetor didático:

$$
x = [3, 4]
$$

Pode-se imaginá-lo como uma seta que sai da origem e chega ao ponto $(3, 4)$: são 3 unidades na horizontal e 4 na vertical.

#### 1. Calcular o comprimento do vetor

A **norma L2** representa o comprimento, ou magnitude, do vetor. Pelo teorema de Pitágoras:

$$
\|x\|_2 = \sqrt{3^2 + 4^2} = \sqrt{25} = 5
$$

Portanto, o vetor possui comprimento 5.

#### 2. Normalizar o vetor

Normalizar significa transformar o vetor em outro de **comprimento 1, mantendo sua direção**.

Para isso, divide-se cada coordenada pela norma:

$$
x_{\text{norm}} = \frac{x}{\|x\|_2}
= \left[\frac{3}{5}, \frac{4}{5}\right]
= [0{,}6,\; 0{,}8]
$$

A nova seta aponta na mesma direção, mas ficou cinco vezes menor. Seu comprimento passa a ser:

$$
\|x_{\text{norm}}\|_2
= \sqrt{0{,}6^2 + 0{,}8^2}
= \sqrt{0{,}36 + 0{,}64}
= 1
$$

#### 3. Entender a relação com a similaridade de cosseno

Na recuperação de documentos, compara-se o embedding da pergunta com os embeddings dos trechos.

A **similaridade de cosseno** compara a direção de dois vetores:

$$
\operatorname{sim}_{\cos}(q, x)
= \frac{q \cdot x}{\|q\|_2 \|x\|_2}
$$

Nessa expressão:

- $q$ é o vetor da pergunta;
- $x$ é o vetor de um trecho;
- $q \cdot x$ é o produto escalar;
- $\|q\|_2$ e $\|x\|_2$ são os comprimentos dos vetores.

Quando **os dois vetores estão normalizados**, suas normas são iguais a 1. Assim:

$$
\operatorname{sim}_{\cos}(q_{\text{norm}}, x_{\text{norm}})
= q_{\text{norm}} \cdot x_{\text{norm}}
$$

Portanto, a similaridade de cosseno pode ser calculada diretamente pelo produto escalar dos vetores normalizados.

#### 4. Calcular um produto escalar

O produto escalar consiste em multiplicar as coordenadas correspondentes e somar os resultados.

Comparando o vetor normalizado consigo mesmo:

$$
[0{,}6,\; 0{,}8] \cdot [0{,}6,\; 0{,}8]
= 0{,}6 \times 0{,}6 + 0{,}8 \times 0{,}8
= 1
$$

A similaridade é 1 porque os vetores possuem direções idênticas.

#### Aplicação no projeto

Normalizam-se tanto os embeddings dos documentos quanto o embedding da pergunta. Isso permite calcular a similaridade de cosseno pelo produto escalar e ordenar os trechos por proximidade no espaço de embeddings.

Para um vetor com centenas de dimensões, o procedimento é o mesmo:

$$
\|x\|_2 = \sqrt{\sum_{i=1}^{d} x_i^2}
\qquad
x_{\text{norm},i} = \frac{x_i}{\|x\|_2}
$$

Aqui, $d$ é o número de dimensões. A normalização pressupõe um vetor não nulo, pois não é possível dividir por uma norma igual a zero.

**Interpretação e limites:** o vetor $[3, 4]$ serve apenas para ilustrar a conta; não representa um embedding real do corpus. A similaridade de cosseno é uma medida de proximidade entre vetores, não uma probabilidade de o trecho ser relevante ou de a resposta estar correta.

In [3]:
# Objetivo: criar chunks e embeddings normalizados.
import numpy as np
from sentence_transformers import SentenceTransformer
NOME_ENCODER = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# Carrega o encoder semântico; device="cpu" preserva VRAM para o modelo gerador.
encoder = SentenceTransformer(NOME_ENCODER, device="cpu")
TAMANHO_CHUNK = min(96, encoder.max_seq_length - 2)
SOBREPOSICAO = 16
chunks = []
for pagina in paginas:
    # encode transforma o texto em IDs; add_special_tokens=False exclui marcadores do modelo.
    tokens = encoder.tokenizer.encode(pagina["texto"], add_special_tokens=False)
    for inicio in range(0, len(tokens), TAMANHO_CHUNK - SOBREPOSICAO):
        # decode reconstrói o trecho textual a partir da janela de IDs.
        texto = encoder.tokenizer.decode(tokens[inicio:inicio + TAMANHO_CHUNK], skip_special_tokens=True)
        if not texto.strip():
            continue
        registro = dict(pagina)
        registro["texto"] = texto
        registro["id"] = hashlib.sha256(f"{pagina['source_id']}:{pagina['page']}:{inicio}:{texto}".encode()).hexdigest()
        chunks.append(registro)
print("Chunks:", len(chunks))
print(chunks[5])
# encode gera um vetor por texto; normalize_embeddings cria vetores unitários, batch_size define lote e show_progress_bar exibe progresso.
vetores = encoder.encode([c["texto"] for c in chunks], normalize_embeddings=True, batch_size=16, show_progress_bar=True)
print("Matriz:", vetores.shape)
print("Normas:", np.linalg.norm(vetores[:5], axis=1))

c:\Users\leova\anaconda3\envs\engenharia_ia_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9467.63it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (675 > 128). Running this sequence through the model will result in indexing errors


Chunks: 990
{'texto': 'successfully to English constituency parsing both with large and limited training data. ∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started the effort to evaluate this idea. Ashish, with Illia, designed and implemented the first Transformer models and has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head', 'source_id': 'attention_is_all_you_need', 'title': 'Attention Is All You Need', 'page': 1, 'url': 'https://arxiv.org/pdf/1706.03762', 'id': '191e5f23a7f3138010f0c0e8278ad3fed77157b053d804a936472803f8c32357'}


Batches: 100%|██████████| 62/62 [00:15<00:00,  3.92it/s]

Matriz: (990, 384)
Normas: [1. 1. 1. 1. 1.]


## 3. Persistir a saída para a próxima etapa

Os chunks com texto e metadados são gravados em JSONL; os embeddings vão para um arquivo NumPy; o manifesto registra encoder e parâmetros de divisão. Assim, o notebook 02 pode carregar os mesmos vetores e construir o Chroma sem recalcular a extração e os embeddings.

O checkpoint desta etapa é conferir quantidade, dimensões, normas dos vetores e rastreabilidade de alguns chunks até o PDF. Só então faz sentido criar o índice de busca.

---

### Por que guardar também o manifesto?

O JSONL preserva o que será recuperado; o arquivo NumPy guarda os vetores; o manifesto registra como ambos foram produzidos. Se o encoder ou os parâmetros de chunking mudarem, o manifesto ajuda a detectar que os vetores antigos precisam ser reconstruídos. Exemplo: 990 chunks exigem uma matriz com 990 linhas; a linha 17 precisa continuar apontando para o mesmo chunk 17 no JSONL.

In [4]:
# Objetivo: salvar corpus, vetores e manifesto.
import numpy as np
import json

caminho_chunks = DIRETORIO_PROCESSADOS / "chunks.jsonl"
caminho_chunks.write_text(
    "\n".join(json.dumps(chunk, ensure_ascii=False) for chunk in chunks) + "\n",
    encoding="utf-8",
)
# np.save grava a matriz NumPy em formato binário para reutilização.
np.save(DIRETORIO_PROCESSADOS / "embeddings.npy", vetores.astype("float32"))
manifesto = {
    "encoder": NOME_ENCODER,
    "tamanho_chunk_tokens": TAMANHO_CHUNK,
    "sobreposicao_tokens": SOBREPOSICAO,
    "quantidade_paginas": len(paginas),
    "quantidade_chunks": len(chunks),
    "dimensao_embedding": int(vetores.shape[1]),
}
(DIRETORIO_PROCESSADOS / "manifesto_corpus.json").write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(manifesto)

{'encoder': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'tamanho_chunk_tokens': 96, 'sobreposicao_tokens': 16, 'quantidade_paginas': 103, 'quantidade_chunks': 990, 'dimensao_embedding': 384}
